# 06 — Author a custom scenario

Subclass `BaseScenario`, implement `calculate_reward` + `check_termination`, plug it into the env. Below: a 'protect the domain controller' scenario where Blue is rewarded for keeping the DC's privilege at `None`.

In [ ]:
!pip install -q 'git+https://github.com/reforcemind/NetForge_RL'

In [ ]:
from netforge_rl.environment.parallel_env import NetForgeRLEnv
from netforge_rl.scenarios.base_scenario import BaseScenario

class ProtectDC(BaseScenario):
    def __init__(self, agents):
        self.agents = agents
    def calculate_reward(self, agent_id, global_state, effect=None):
        dc = next((h for h in global_state.all_hosts.values() if h.is_domain_controller), None)
        if dc is None:
            return 0.0
        # Blue gets +1/tick while DC is clean; Red gets +1/tick once they own it.
        owned = dc.privilege in ('User', 'Root')
        if 'blue' in agent_id.lower():
            return -10.0 if owned else 1.0
        return 10.0 if owned else -0.1
    def check_termination(self, global_state):
        return {a: False for a in self.agents}

env = NetForgeRLEnv({'scenario_type': 'ransomware', 'max_ticks': 30})
env.reset(seed=0)
env.scenario = ProtectDC(env.possible_agents)  # swap in the custom scenario

import numpy as np
rng = np.random.default_rng(0)
totals = {a: 0.0 for a in env.possible_agents}
while env.agents:
    acts = {a: np.array([rng.integers(0, n) for n in env.action_space(a).nvec], dtype=np.int64) for a in env.agents}
    _, r, term, trunc, _ = env.step(acts)
    for a, v in r.items():
        totals[a] += v
    if all(term.values()) or all(trunc.values()):
        break
totals